In [ ]:
# Import required libraries
import torch
import torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt
import requests
from PIL import Image
import io
import math

# Define the custom Gaussian blur function (as provided)
def apply_gaussian_blur(images, kernel_size=21, sigma=7.0):
    # Create a Gaussian kernel
    def gaussian_kernel(kernel_size, sigma=1.0):
        x = torch.arange(kernel_size).float() - (kernel_size - 1) / 2
        x = x.view(1, -1).expand(kernel_size, -1)
        y = torch.arange(kernel_size).float() - (kernel_size - 1) / 2
        y = y.view(-1, 1).expand(-1, kernel_size)
        coefficient = 1.0 / (2 * math.pi * sigma**2)
        kernel = coefficient * torch.exp(-(x**2 + y**2) / (2 * sigma**2))
        return kernel / torch.sum(kernel)
    
    # Create a 2D Gaussian kernel
    kernel = gaussian_kernel(kernel_size, sigma)
    
    # Expand to 3 channels for RGB
    kernel = kernel.view(1, 1, kernel_size, kernel_size)
    kernel = kernel.repeat(3, 1, 1, 1)
    
    # Apply padding to maintain the same image size
    padding = (kernel_size - 1) // 2
    
    # Apply the kernel to each channel separately to maintain correct color
    blurred_images = F.conv2d(
        images, 
        weight=kernel.to(images.device), 
        padding=padding,
        groups=3  # Apply separately to each channel
    )
    
    return blurred_images

# Load the image from a URL (Albert Einstein, public domain)
url = "https://upload.wikimedia.org/wikipedia/commons/3/3e/Albert_Einstein_Head.jpg"
response = requests.get(url)
image = Image.open(io.BytesIO(response.content))

# Convert the PIL image to a PyTorch tensor and add a batch dimension
transform = torchvision.transforms.ToTensor()  # Converts to [0,1] range, shape (C, H, W)
image_tensor = transform(image).unsqueeze(0)  # Shape becomes (1, C, H, W)

# Apply Gaussian blur
blurred_image_tensor = apply_gaussian_blur(image_tensor)

# Clamp the output to ensure values stay in [0,1] range
blurred_image_tensor = torch.clamp(blurred_image_tensor, 0, 1)

# Remove the batch dimension for visualization
original_tensor = image_tensor.squeeze(0)  # Shape (C, H, W)
blurred_tensor = blurred_image_tensor.squeeze(0)  # Shape (C, H, W)

# Convert tensors to numpy arrays and permute dimensions for Matplotlib (H, W, C)
original_image = original_tensor.permute(1, 2, 0).numpy()
blurred_image = blurred_tensor.permute(1, 2, 0).numpy()

# Display the original and blurred images side by side
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(original_image)
axes[0].set_title("Original Image")
axes[0].axis('off')
axes[1].imshow(blurred_image)
axes[1].set_title("Blurred Image (kernel=21, sigma=7.0)")
axes[1].axis('off')
plt.show()